## 5.2 랭그래프 기본 개념 이해하고 적용하기

In [1]:
# .env 파일에 저장된 환경변수(GOOGLE_API_KEY 등)를 불러오는 도구를 가져옵니다.
from dotenv import load_dotenv

# 환경변수를 실제로 메모리에 로드합니다. 성공하면 True가 출력됩니다.
load_dotenv()

True

### 5.2.1 [실습] 그래프의 상태 정의하기

In [2]:
# TypedDict: 딕셔너리(사전형 데이터)에 어떤 이름(Key)과 어떤 자료형(Type)이 들어갈지
# 미리 '설계도(양식)'를 만들 때 사용하는 도구입니다.
from typing_extensions import TypedDict

# State(상태): 랭그래프에서 노드들이 서로 주고받는 '공유 메모장(바구니)'입니다.
# 여기서는 'messages'라는 이름의 칸에 문자열 리스트(list[str])를 담겠다고 선언했습니다.
class State(TypedDict):
    messages: list[str]

In [3]:
# [예제] TypedDict를 사용해 사용자(User) 데이터 양식 정의하기
from typing import TypedDict

# User라는 이름의 딕셔너리 양식을 만듭니다.
# - id는 정수(int), name은 문자열(str), email은 문자열(str)이어야 한다고 안내합니다.
class User(TypedDict):
    id: int
    name: str
    email: str

# 위에서 정의한 User 양식에 맞춰 실제 데이터를 생성합니다.
user1: User = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 'nayeon_park', 'email': 'example@gmail.com'}


In [4]:
# [주의] TypedDict의 특징: 실행 중(Runtime)에는 타입을 강제로 검사하지 않습니다!
# 설계도에서 name은 문자열(str)이라고 했지만, 숫자(123)를 넣어도 에러 없이 실행됩니다.
# (비유: 서류 양식에 '이름(한글)'이라고 적혀있지만, 숫자를 적어도 접수처에서 막지는 않는 것과 같습니다.)
user1: User = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 123, 'email': 'example@gmail.com'}


In [5]:
# [비교] Pydantic의 BaseModel: 실행 중에도 데이터 타입을 깐깐하게 검사하는 도구입니다.
from pydantic import BaseModel

# BaseModel을 상속받아 User 설계도를 만듭니다.
class User(BaseModel):
    id: int
    name: str
    email: str

user_data = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}

# **user_data는 딕셔너리 안의 내용물을 풀어서(id=1, name='nayeon_park', ...) 넣어주는 파이썬 문법입니다.
user1 = User(**user_data)
print(user1)

id=1 name='nayeon_park' email='example@gmail.com'


In [6]:
# Pydantic의 BaseModel을 쓰면, name에 숫자(123)가 들어왔을 때
# 'ValidationError(유효성 검사 에러: 문자열이어야 합니다!)'를 발생시켜 실수를 즉시 잡아줍니다.
# (참고: 랭그래프에서는 가볍고 빠른 처리를 위해 주로 TypedDict를 사용하지만, 엄격한 검사가 필요할 땐 Pydantic도 지원합니다.)
user_data = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}

user = User(**user_data)
print(user)

ValidationError: 1 validation error for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

### 5.2.2 [실습] 그래프의 상태에 리듀서 함수 추가하기

In [7]:
# Annotated: 자료형 옆에 '추가 규칙(메타데이터)'을 덧붙일 때 사용하는 도구입니다.
from typing import TypedDict, Annotated

# 리듀서(Reducer) 함수 정의: 기존 데이터(left)와 새 데이터(right)를 어떻게 합칠지 결정합니다.
# 만약 리듀서를 지정하지 않으면, 새로운 값이 들어올 때 기존 값을 '덮어쓰기(삭제 후 교체)' 해버립니다.
# 대화 기록은 사라지면 안 되고 계속 이어붙여야 하므로 두 리스트를 더하는(+) 함수를 만듭니다.
def add(left, right):
    return left + right

# messages 칸에 새로운 리스트가 들어오면 덮어쓰지 말고 add 함수로 '이어붙여라!'라고 지정합니다.
class State(TypedDict):
    messages: Annotated[list[str], add]

In [8]:
# 랭그래프가 기본 제공하는 똑똑한 메시지 전용 리듀서 'add_messages' 알아보기
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

# 고유 번호(id)가 서로 다른 두 메시지('1'번과 '2'번)를 준비합니다.
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]

# ID가 서로 다르면 두 메시지를 순서대로 합쳐서 하나의 대화 목록으로 만듭니다.
add_messages(msgs1, msgs2)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1'),
 AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='2')]

In [9]:
# [add_messages의 특별한 기능] 만약 이미 있는 메시지 ID('1')와 똑같은 ID('1')로 새 메시지가 들어오면?
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [HumanMessage(content="Hello again", id="1")]

# 단순한 덧셈(+)과 달리, 같은 ID를 가진 기존 메시지를 새 내용('Hello again')으로 똑똑하게 수정(업데이트)해 줍니다!
add_messages(msgs1, msgs2)

[HumanMessage(content='Hello again', additional_kwargs={}, response_metadata={}, id='1')]

In [10]:
# 실전 에이전트에서 가장 많이 사용하는 표준 상태(State) 정의 패턴입니다.
# AnyMessage: HumanMessage(사용자), AIMessage(AI), SystemMessage(시스템) 등 모든 종류의 메시지를 뜻합니다.
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

### 5.2.3 [실습] 그래프의 실행 단위, 노드 추가하기

In [11]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph

# 1. 상태(공유 메모장) 양식 정의하기
class State(TypedDict):
    messages: Annotated[list[str], add]

# 2. StateGraph(상태 기반 그래프) 설계도 도화지 펼치기
graph = StateGraph(State)

# 3. 실제 작업을 수행할 '노드(Node, 작업자 함수)' 정의하기
# [ 1 ] 노드는 현재 상태(state)를 입력으로 받습니다.
def chatbot(state: State): # [ 1 ]
    question = state["messages"]
    # [ 2 ] 받은 질문을 바탕으로 안내 문구를 만듭니다.
    answer = f"사용자 입력을 그대로 반환하는 챗봇입니다. {question} 라는 질문을 받았습니다." # [ 2 ]
    # [ 3 ] 상태(State)에 업데이트할 내용을 딕셔너리 형태로 반환합니다. (리듀서 add에 의해 기존 목록 뒤에 추가됨)
    return {"messages": [answer]} # [ 3 ]

# [ 4 ] 그래프 도화지에 "chatbot"이라는 이름으로 chatbot 함수를 노드(역/정거장)로 등록합니다.
graph.add_node("chatbot", chatbot) # [ 4 ]

### 5.2.5 [실습] 그래프에 조건부 엣지 추가하기

In [12]:
# 엣지(Edge): 노드와 노드를 연결하는 '화살표(이동 경로)'입니다.
# START는 그래프의 출발점, END는 그래프의 도착점(종료)을 의미합니다.
from langgraph.graph import START, END

# 출발점(START)에서 시작하면 무조건 "chatbot" 노드로 이동하라는 화살표를 그립니다.
graph.add_edge(START, "chatbot")
# "chatbot" 노드가 끝나면 무조건 도착점(END)으로 이동해 종료하라는 화살표를 그립니다.
graph.add_edge("chatbot", END)

In [13]:
# 일반 엣지 예시: "node_a" 작업이 끝나면 항상 "node_b"로 이동하도록 연결할 때 이렇게 작성합니다.
graph.add_edge("node_a", "node_b")

**조건부 엣지**

In [ ]:
# 조건부 엣지(Conditional Edge): 상황에 따라 갈림길(분기점)을 선택하는 똑똑한 화살표입니다.
# 1. 갈림길 안내원(라우팅 함수) 정의: 마지막 메시지의 글자 수가 1000자를 넘으면 True, 아니면 False를 반환합니다.
def routing_function(state: State):
    if len(state["messages"][-1]) > 1000 :
        return True
    return False

# 2. 조건부 화살표 연결하기:
# "chatbot" 노드가 끝난 직후, routing_function을 실행해 결과를 확인합니다.
# 결과가 True이면 "summary"(요약 노드)로 가고, False이면 END(종료)로 이동합니다.
graph.add_conditional_edges(
	"chatbot",
	routing_function,
	{True: "summary", False: END}
)